In [ ]:
import pandas as pd
from pathlib import Path

# =========================================================
# LDBC SNB CROSS-SCALE COMPARISON — CORRECTED VERSION
# =========================================================
# Aligned with comparison_sfs_imdb.ipynb:
#
# 1) Reads one corrected per-scale analysis file per result folder.
# 2) Filters explicitly by run_phase.
# 3) Computes cross-scale summary.
# 4) Includes near-best preservation when available.
# 5) Exports final cross-scale artifacts.
# =========================================================

# ---------------------------------------------------------
# 1) Configuration
# ---------------------------------------------------------

base_results = Path(
    "/path/to/schemalens/ldbc_snb/results"
)

paths = {
    "sf0.1": base_results / "ldbc_snb_sf0_1_full_fiben_format_clean" / "ldbc_snb_analysis_outputs" / "ldbc_snb_summary_hot_by_scale.csv",
    "sf1":   base_results / "ldbc_snb_sf1_full_fiben_format_clean" / "ldbc_snb_analysis_outputs" / "ldbc_snb_summary_hot_by_scale.csv",
    "sf3":   base_results / "ldbc_snb_sf3_full_fiben_format_clean" / "ldbc_snb_analysis_outputs" / "ldbc_snb_summary_hot_by_scale.csv",
}

fallback_filename = "schemalens_reduction_analysis_hot.csv"

selected_run_phase = "hot"

# ---------------------------------------------------------
# 2) Read all per-scale corrected analysis files
# ---------------------------------------------------------

dfs = []

for scale, path in paths.items():
    if not path.exists():
        fallback_path = path.parent.parent / fallback_filename
        if fallback_path.exists():
            path = fallback_path
        else:
            raise FileNotFoundError(
                f"Could not find analysis file for {scale}.\n"
                f"Tried:\n- {path}\n- {fallback_path}"
            )

    print(f"Reading {scale}: {path}")

    df = pd.read_csv(path)

    if "scale_label" not in df.columns:
        df["scale_label"] = scale

    # Force expected scale label when file has one scale only
    if df["scale_label"].nunique() == 1:
        df["scale_label"] = scale

    if "run_phase" in df.columns:
        df = df[df["run_phase"] == selected_run_phase].copy()
    else:
        df["run_phase"] = selected_run_phase

    dfs.append(df)

cross_scale_schemalens_df = pd.concat(dfs, ignore_index=True)

# ---------------------------------------------------------
# 3) Main cross-scale summary
# ---------------------------------------------------------

agg_dict = dict(
    n_queries=("query_name", "nunique"),
    avg_DSR=("DSR", "mean"),
    top1_preservation=("top1_preserved_by_activated", "mean"),
    mean_activated_regret=("activated_regret", "mean"),
    mean_primary_regret=("primary_regret", lambda x: x.dropna().mean()),
    primary_winners=("best_group", lambda x: int((x == "primary").sum())),
    secondary_winners=("best_group", lambda x: int((x == "secondary_affected").sum())),
    control_winners=("best_group", lambda x: int((x == "control").sum())),
)

if "near_best_preserved_by_activated" in cross_scale_schemalens_df.columns:
    agg_dict["near_best_preservation"] = ("near_best_preserved_by_activated", "mean")

summary_df = (
    cross_scale_schemalens_df
    .groupby("scale_label")
    .agg(**agg_dict)
    .reset_index()
)

scale_order = {"sf0.1": 0, "sf1": 1, "sf3": 2, "sf10": 3}
summary_df["scale_order"] = summary_df["scale_label"].map(scale_order).fillna(999)
summary_df = summary_df.sort_values("scale_order").drop(columns=["scale_order"])

display(summary_df)

# ---------------------------------------------------------
# 4) Secondary winners table
# ---------------------------------------------------------

secondary_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    secondary_cols.insert(1, "run_phase")

secondary_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "secondary_affected"
][secondary_cols].sort_values(["scale_label", "official_id"])

display(secondary_df)

# ---------------------------------------------------------
# 5) Control winners table
# ---------------------------------------------------------

control_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_design_pattern",
    "best_p95_ms",
    "best_primary_config",
    "best_primary_p95_ms",
    "primary_regret",
    "activated_regret",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    control_cols.insert(1, "run_phase")

control_df = cross_scale_schemalens_df[
    cross_scale_schemalens_df["best_group"] == "control"
][control_cols].sort_values(["scale_label", "official_id"])

display(control_df)

# ---------------------------------------------------------
# 6) Best-by-scale overview
# ---------------------------------------------------------

best_by_scale_cols = [
    "scale_label",
    "official_id",
    "query_name",
    "best_config",
    "best_group",
    "best_design_pattern",
    "best_p95_ms",
]

if "run_phase" in cross_scale_schemalens_df.columns:
    best_by_scale_cols.insert(1, "run_phase")

best_by_scale_df = (
    cross_scale_schemalens_df[best_by_scale_cols]
    .sort_values(["scale_label", "official_id"])
    .reset_index(drop=True)
)

display(best_by_scale_df)

# ---------------------------------------------------------
# 7) p95 pivot across scales
# ---------------------------------------------------------

p95_pivot_df = (
    cross_scale_schemalens_df
    .pivot_table(
        index=["official_id", "query_name"],
        columns="scale_label",
        values="best_p95_ms",
        aggfunc="first"
    )
    .reset_index()
)

if "sf0.1" in p95_pivot_df.columns and "sf1" in p95_pivot_df.columns:
    p95_pivot_df["sf1_vs_sf0_1_ratio"] = p95_pivot_df["sf1"] / p95_pivot_df["sf0.1"]

if "sf1" in p95_pivot_df.columns and "sf3" in p95_pivot_df.columns:
    p95_pivot_df["sf3_vs_sf1_ratio"] = p95_pivot_df["sf3"] / p95_pivot_df["sf1"]

if "sf0.1" in p95_pivot_df.columns and "sf3" in p95_pivot_df.columns:
    p95_pivot_df["sf3_vs_sf0_1_ratio"] = p95_pivot_df["sf3"] / p95_pivot_df["sf0.1"]

display(p95_pivot_df)

# ---------------------------------------------------------
# 8) Optional near-best summary
# ---------------------------------------------------------

if "near_best_preserved_by_activated" in cross_scale_schemalens_df.columns:
    near_best_summary_df = (
        cross_scale_schemalens_df
        .groupby("scale_label")
        .agg(
            near_best_preservation=("near_best_preserved_by_activated", "mean"),
            mean_activated_regret=("activated_regret", "mean"),
        )
        .reset_index()
    )

    near_best_summary_df["scale_order"] = near_best_summary_df["scale_label"].map(scale_order).fillna(999)
    near_best_summary_df = near_best_summary_df.sort_values("scale_order").drop(columns=["scale_order"])

    display(near_best_summary_df)
else:
    near_best_summary_df = pd.DataFrame()

# ---------------------------------------------------------
# 9) Save outputs
# ---------------------------------------------------------

out_dir = base_results / "cross_scale_analysis_final_corrected"
out_dir.mkdir(parents=True, exist_ok=True)

cross_scale_schemalens_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_schemalens_analysis_final.csv",
    index=False
)

summary_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_summary_final.csv",
    index=False
)

secondary_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_secondary_winners_final.csv",
    index=False
)

control_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_control_winners_final.csv",
    index=False
)

best_by_scale_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_best_by_scale_final.csv",
    index=False
)

p95_pivot_df.to_csv(
    out_dir / "ldbc_snb_cross_scale_best_p95_by_query_final.csv",
    index=False
)

if not near_best_summary_df.empty:
    near_best_summary_df.to_csv(
        out_dir / "ldbc_snb_cross_scale_near_best_summary_final.csv",
        index=False
    )

print("Saved to:", out_dir)